# Import Statements

In [ ]:
import os
import custom_cmap
from PIL import Image
from IPython.display import display
 
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd

from moria import reduce s 
from pathlib import Path

from astropy.visualization import PercentileInterval
from astropy.io import fits
from astropy.visualization import LogStretch, ImageNormalize
import plotly.express as px
import numpy as np
from astropy.visualization import ImageNormalize, AsinhStretch, SqrtStretch, LogStretch, PowerStretch

plt.rcParams.update({'font.size':25})
plt.rc('text', usetex=True)
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

# Understanding the main directory. 

The data directory should be organized as follows. You can look at the sample data folder to understand the directory strucutre you need.

<li>00.DATA</li>
<li>01.XYM</li>
<li>02.CMD</li>
<li>03.LOC_TRANS</li>
<li>04.PSF_EXTRACT</li>
<li>05.COORD_TRANS(OPTIONAL)</li>
<li>06.FIT</li>
<li>07.CALIBRATION</li>

All the necessary scripts are included in the sample data directory. The simplest way to run MORIA on your target is to copy the entirety of the "data" folder to wherever you wish to conduct your analysis. Start by putting your exposures in "data/00.DATA", then beginning with the demo. 

After you finish running this notebook, you will be able to use the "_flc" exposures to generate an output image stack in the F814W and F606W HST filters.

NOTE: chmod +x program.src is a useful command to use whenever permissions are denied for a script. If any of these scripts fail, each folder in the list above (00.DATA, 01.XYM, 02.CMD, etc.) has an output "log file" for the Fortran scripts we run. If the "log file" for the step you ran shows a "permission denied" error, it is very likely that you need to run chmod _x program.src for each ".src" file used in this pipeline.

In [ ]:
#This is the directory where you are processing your data. This does not point at MORIA.
directory = os.getcwd()

# Data Preparation 

MORIA runs the "Makefile" in fortran_compile directly upon pip installation.

 If you have different fortran dependincies, you will have to edit the Makefile.txt in fortran_compile, and then uncomment the code_directory to point to src/fortran_compile. Once you finish compiling the files above, run this cell below to put the fortran scripts in their appropriate directories:

In [ ]:

reduce.data_prep_early(directory)

# STEP 1

<li>Download the _flc files. Place them appropriately in the F814W and F606W directories in 00.DATA.</li>
<li> The cell below will convert the _flc files to _WJ2 files using run_convert_C1K1C.src</li>

These "_flc" exposures include the charge-transfer efficiency (CTE) corrections. MORIA first converts these "_flc" exposures into "_WJ2" files to convert to a full-chip co-ordiante system that are placed into appropriate reference geometry for subsequent steps. 

In [ ]:
reduce.run_xgf_conversion(directory)



# Step 2 

Prepare the IN.* files taken in by the .src scripts in 01.XYM and 02.CMD directories. This step is important since the IN.* files in each directory are used to provide input to the backend Fortran scripts.

In [ ]:
reduce.data_prep(directory)

# Creature MATCHUP files

In the cell below we will achieve the following outcomes:

<li>We will use respective PSF files for each filter to create a .XYM file for each exposure. A library PSF model is used to produce ".XYM" files, containing the star positions and magnitudes for each HST image exposure.</li>
<li>Generate output files TRANS.xym2mat, along with 16 MAT.0 files. Note that the first line of the IN.xym2mat and files beginning with 00 just defines the reference coordinates to be used for the output, so the 00 file does not need to be a *_WJ2.xym file.</li>
<li> Find all the objects that could be found in at least 12 out of 16 exposures. </li>

In [ ]:
reduce.matchup_files(directory)

Let's open the MATCHUP file to inspect it further. The MATCHUP file is stored in 01.XYM/F814W for the F814W filter and 01.XYM/F606W for the F606W filter

THe MATCHUP file we will open will be for the F814W folder. You can uncomment the filename_606W line in the cell below to look at the F606W MATCHUP file.

In [ ]:
filename = Path(directory).resolve()/f"01.XYM/dex_no_gaia_STEP08_A.xyvieeee"
cols = ["xbar", "ybar", "mv", "mi"]
df = pd.read_csv(filename, sep=r"\s+", comment="#", usecols = [0,1,2,3], header=None, names=cols)

In [ ]:
df

From the MATCHUP file above, here are the most important things to note:

1. xbar and ybar correspond to the x and y star positions (in pixel coordinates) for stars in the HST images.
2. mv is the F606W V magnitude of the stars.
2. mi is the F814W I magnitude of the stars.
Using the MATCHUP files, we have also created an HST output image stack in the F606W and F814W filters. Let's look at them.

You can view the output stack in the jupyter notebook here. This is a down-scaled version because the .fits file is too large to open in the notebook directly. The output stacks for both filters are in 01.XYM. 

We open the output stack for F814W below. If you want to see the upscaled true version of the FITS file, use ds9 on your terminal.

The scaling can be adjusted depending on your preference by changing the argument given to "ImageNormalize" in the cell below.

In [ ]:
filename_814W = Path(directory).resolve() / "01.XYM/dex_no_gaia_STEP10B_F814W.fits"
filename_606W = Path(directory).resolve() / "01.XYM/dex_no_gaia_STEP10A_F606W.fits"

# Full stacks are often 10k+ pixels; loading + Plotly serializes the entire array and kills the kernel.
# Keep only a strided preview in RAM for visualization (full FITS on disk is unchanged).
MAX_PREVIEW_SIDE = 1536

def read_stack_preview(path, max_side=MAX_PREVIEW_SIDE):
    with fits.open(path, memmap=True) as hdul:
        raw = hdul[-1].data
        ny, nx = int(raw.shape[0]), int(raw.shape[1])
        step = max(int(np.ceil(max(ny, nx) / max_side)), 1)
        # Small contiguous copy; stride read from memmap avoids holding the full image in memory.
        preview = np.array(raw[::step, ::step], dtype=np.float32, copy=True)
    preview = np.nan_to_num(preview, nan=0.0, posinf=0.0, neginf=0.0)
    return preview, step, (ny, nx)

preview, stride, full_shape = read_stack_preview(filename_814W)
norm = ImageNormalize(preview, stretch=AsinhStretch(0.001))
scaled = norm(preview)

cmap = custom_cmap.mpl_nb()
fig, ax = plt.subplots(figsize=(11, 11))
im = ax.imshow(scaled, origin="lower", cmap=cmap, interpolation="nearest")
ax.set_title(f"Output Stack (F814W)")
ax.set_xlabel("column (preview pixels)")
ax.set_ylabel("row (preview pixels)")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02, label="Scaled intensity")
plt.show()
